# Adding Parameters to Sets &mdash; Best Practices

In `UnichartNotebook`, **all sets share one combined DataFrame**; each `Dataset` is a
view into the rows tagged with its set id. That single fact drives every recommendation
below: a write goes into the shared frame *masked to one set's rows*, so **one column name
can hold different values for different sets** &mdash; which is exactly what lets you compute a
parameter differently per set and still plot or table them together afterward.

### The decision rule (use this to pick a method)

| You want to&hellip; | Use | Why |
|---|---|---|
| compute a column from a set's **own** columns | `ds['col'] = expr` | the supported per-set write; reads/writes that set's rows |
| apply **one uniform formula** to every set at once | `nb.add_column(name, fn)` | a single vectorized pass over the whole combined frame |
| push an **externally-computed block** of values to specific set(s) | `nb.set_column(sets, col, vals)` | targets chosen sets; NaN-fills the rest |

The rest of the notebook works through each, with runnable checks (`PASS` prints) that
assert the results against hand-computed values.

## 0. Setup &mdash; three engine datasets

Three engine variants logged across an `N1` (fan-speed %) sweep, with fuel flow `FF` and
exhaust temp `EGT`. The variants differ, which is what makes per-set formulas meaningful.
We keep every set **query-free** so the arithmetic checks below are exact.

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook

def engine(n1, ff, egt):
    return pd.DataFrame({'N1': np.array(n1, float),
                         'FF': np.array(ff, float),
                         'EGT': np.array(egt, float)})

uc = UnichartNotebook()
uc.load(engine([40, 55, 70, 85, 100], [900, 1500, 2300, 3400, 4800], [420, 480, 560, 650, 760]),
        title='Eng-A (turbofan)')   # Set 0
uc.load(engine([42, 56, 72, 88, 100], [950, 1600, 2450, 3600, 5000], [430, 495, 575, 670, 780]),
        title='Eng-B (turbofan)')   # Set 1
uc.load(engine([45, 60, 75, 90, 100], [1100, 1800, 2700, 3900, 5300], [460, 540, 630, 730, 850]),
        title='Eng-C (turbojet)')   # Set 2

uc.sets[0].df

UniChart Notebook Environment Initialized.
Loaded Set 0: Eng-A (turbofan)
Loaded Set 1: Eng-B (turbofan)
Loaded Set 2: Eng-C (turbojet)


,N1,FF,EGT
0,40.0,900.0,420.0
1,55.0,1500.0,480.0
2,70.0,2300.0,560.0
3,85.0,3400.0,650.0
4,100.0,4800.0,760.0


## 1. The blessed write: `ds['col'] = ...`

From the `Dataset.__setitem__` docstring:

> *Writing through `ds.df[...]` does NOT persist, because `df` returns a fresh slice (a copy)
> each call. This is the supported way to add or modify columns, e.g. `ds['NEW'] = ds['A'] * 2`.*

So always assign through the dataset itself (`ds['col'] = ...`). For contrast, the cell below
shows that writing through the `.df` view is silently lost.

In [2]:
ds = uc.sets[0]

# CORRECT: persists into the combined frame for this set.
ds['FF_PER_N1'] = ds['FF'] / ds['N1']

# WRONG: ds.df is a fresh copy, so this assignment evaporates.
ds.df['GHOST'] = -1

cols = list(uc.sets[0].df.columns)
assert 'FF_PER_N1' in cols, 'ds[...] should persist'
assert 'GHOST' not in cols, 'ds.df[...] should NOT persist'
print('persisted columns:', cols)
print('PASS - ds[...] persists; ds.df[...] does not.')

persisted columns: ['N1', 'FF', 'EGT', 'FF_PER_N1']
PASS - ds[...] persists; ds.df[...] does not.


/tmp/ipykernel_75558/2200296568.py:7: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  ds.df['GHOST'] = -1


## 2. Same formula for every set &mdash; two clean idioms

When the *same* expression applies to all sets, you have two good options:

1. **Loop with `ds[...]`** &mdash; each set computes from its **own** columns. Readable, and the
   natural choice when the formula references columns by name.
2. **`add_column(name, fn)`** &mdash; one vectorized pass where `fn` receives the whole combined
   frame. Fewer lines when the formula is a simple vectorized expression.

Both produce the identical column; pick whichever reads better for the formula at hand.

In [3]:
# Idiom 1: loop, each set from its own columns.
for d in uc.sets:
    d['FF_PER_N1'] = d['FF'] / d['N1']      # specific fuel-flow proxy

# Idiom 2: one vectorized pass over the combined frame (same result, all sets).
uc.add_column('EGT_K', lambda cdf: cdf['EGT'] + 273.15)   # degC -> Kelvin

for d in uc.sets:
    assert np.allclose(d['FF_PER_N1'], d['FF'] / d['N1'])
    assert np.allclose(d['EGT_K'],     d['EGT'] + 273.15)
print('PASS - loop and add_column both populate every set consistently.')

PASS - loop and add_column both populate every set consistently.


## 3. **Different** formula per set &mdash; the core pattern

This is the main event. Because each `ds['col'] = ...` write is masked into the shared frame
by set id, you can give the **same column name** a **different formula per set**, and the
values coexist &mdash; ready to plot/table side by side.

**Best practice:** keep the per-set parameters in a small dict keyed by set index (or title),
then loop. This keeps the rule in one readable place instead of scattering magic numbers.

Here each engine has its own thrust calibration, `THRUST = k * N1**2`:

In [4]:
# Engine-specific calibration constants, keyed by set index.
k_thrust = {0: 1.00, 1: 1.18, 2: 0.92}

for d in uc.sets:
    d['THRUST'] = k_thrust[d.index] * d['N1'] ** 2

# Same column name 'THRUST', but each set followed its own constant:
for d in uc.sets:
    print(f'{d.title:18s} THRUST = {np.round(d["THRUST"].to_numpy(), 1)}')

assert np.allclose(uc.sets[0]['THRUST'], 1.00 * uc.sets[0]['N1'] ** 2)
assert np.allclose(uc.sets[1]['THRUST'], 1.18 * uc.sets[1]['N1'] ** 2)
assert np.allclose(uc.sets[2]['THRUST'], 0.92 * uc.sets[2]['N1'] ** 2)
print('PASS - one column name, a different per-set formula in each.')

Eng-A (turbofan)   THRUST = [ 1600.  3025.  4900.  7225. 10000.]
Eng-B (turbofan)   THRUST = [ 2081.5  3700.5  6117.1  9137.9 11800. ]
Eng-C (turbojet)   THRUST = [1863. 3312. 5175. 7452. 9200.]
PASS - one column name, a different per-set formula in each.


### Branching on a set property

When the rule is categorical rather than a per-set constant, branch on something about the
set &mdash; e.g. its `title`. Here turbojets get a hotter EGT-margin reference than turbofans,
so the **derived `EGT_MARGIN` uses a different limit per engine type**.

In [5]:
egt_limit = {'turbofan': 800.0, 'turbojet': 900.0}

for d in uc.sets:
    kind = 'turbojet' if 'turbojet' in d.title else 'turbofan'
    d['EGT_MARGIN'] = egt_limit[kind] - d['EGT']     # headroom to the type's limit

for d in uc.sets:
    kind = 'turbojet' if 'turbojet' in d.title else 'turbofan'
    assert np.allclose(d['EGT_MARGIN'], egt_limit[kind] - d['EGT'])
    print(f'{d.title:18s} ({kind:8s}) margin = {np.round(d["EGT_MARGIN"].to_numpy(), 1)}')
print('PASS - per-set formula selected by a set property (title).')

Eng-A (turbofan)   (turbofan) margin = [380. 320. 240. 150.  40.]
Eng-B (turbofan)   (turbofan) margin = [370. 305. 225. 130.  20.]
Eng-C (turbojet)   (turbojet) margin = [440. 360. 270. 170.  50.]
PASS - per-set formula selected by a set property (title).


## 4. Pushing values to specific sets &mdash; `set_column`

Use `set_column(sets, col, values)` to write an **externally-computed block** (a measured array,
a lookup result, a constant tag) into chosen set(s). Non-target sets are NaN-filled, so the
column still exists frame-wide. The value is a scalar or an array sized to the targets' rows.

In [6]:
# A bench-measured correction for Eng-A and Eng-C. An array value must match the
# *total* rows of the selected sets, laid out in set order (Eng-A's 5 rows, then
# Eng-C's 5) -- so here we provide 10 values. (A scalar would broadcast to all of them.)
corr = [0.97, 0.98, 0.99, 1.00, 1.01]
uc.set_column([0, 2], 'BENCH_CORR', corr + corr)   # 10 values = 5 (Eng-A) + 5 (Eng-C)

# A scalar fleet tag on just Eng-B.
uc.set_column(1, 'FLEET', 'reserve')

print('Eng-A BENCH_CORR:', list(uc.sets[0]['BENCH_CORR']))
print('Eng-B BENCH_CORR:', list(uc.sets[1]['BENCH_CORR']), '   <- non-target, NaN-filled')
print('Eng-C BENCH_CORR:', list(uc.sets[2]['BENCH_CORR']))
print('Eng-B FLEET     :', list(uc.sets[1]['FLEET']))

assert np.allclose(uc.sets[0]['BENCH_CORR'], corr)
assert np.allclose(uc.sets[2]['BENCH_CORR'], corr)
assert uc.sets[1]['BENCH_CORR'].isna().all()
assert (uc.sets[1]['FLEET'] == 'reserve').all()
print('PASS - set_column targets chosen sets (in set order) and NaN-fills the rest.')

Eng-A BENCH_CORR: [0.97, 0.98, 0.99, 1.0, 1.01]
Eng-B BENCH_CORR: [<NA>, <NA>, <NA>, <NA>, <NA>]    <- non-target, NaN-filled
Eng-C BENCH_CORR: [0.97, 0.98, 0.99, 1.0, 1.01]
Eng-B FLEET     : ['reserve', 'reserve', 'reserve', 'reserve', 'reserve']
PASS - set_column targets chosen sets (in set order) and NaN-fills the rest.


## 5. Payoff &mdash; the per-set parameter plots together

Because every set wrote its `THRUST` under the same name, a single `plot('N1', 'THRUST')` draws
all three calibrated curves at once &mdash; the per-set formulas diverge exactly as configured
(Eng-B highest at `k=1.18`, Eng-C lowest at `k=0.92`).

In [7]:
uc.select('all')
uc.marker('all', 'o'); uc.linestyle('all', '-')
uc.plot('N1', 'THRUST', suptitle='Per-set THRUST calibration (same column, different formula)')

## 6. A quick look at the assembled frame

`table()` confirms the derived parameters now live alongside the originals for every set.

In [8]:
uc.select('all')
uc.table(cols=['N1', 'THRUST', 'FF_PER_N1', 'EGT_MARGIN'], output='df')

,Set,Dataset,N1,THRUST,FF_PER_N1,EGT_MARGIN
0,0,Eng-A (turbofan),40.0,1600.00,22.500000,380.0
1,0,Eng-A (turbofan),55.0,3025.00,27.272727,320.0
2,0,Eng-A (turbofan),70.0,4900.00,32.857143,240.0
3,0,Eng-A (turbofan),85.0,7225.00,40.000000,150.0
4,0,Eng-A (turbofan),100.0,10000.00,48.000000,40.0
5,1,Eng-B (turbofan),42.0,2081.52,22.619048,370.0
6,1,Eng-B (turbofan),56.0,3700.48,28.571429,305.0
7,1,Eng-B (turbofan),72.0,6117.12,34.027778,225.0
8,1,Eng-B (turbofan),88.0,9137.92,40.909091,130.0
9,1,Eng-B (turbofan),100.0,11800.00,50.000000,20.0


---
### Summary &mdash; best practices

- **Write through `ds['col'] = ...`**, never `ds.df['col'] = ...` (the latter is a copy and is lost).
- **Per-set from own columns:** `ds['col'] = expr` in a loop &mdash; the default, most readable path.
- **One uniform formula everywhere:** `add_column(name, fn)` for a single vectorized pass.
- **External block into chosen sets:** `set_column(sets, col, vals)`; others are NaN-filled.
- **Different per set:** keep constants/limits in a dict keyed by index or title and loop. The
  shared-frame masked write lets one column name carry per-set-different values, so they plot
  and tabulate together with no extra bookkeeping.